# Probability Distributions and the PMF

This notebook introduces the **probability distribution** of a random variable — all its possible values on one axis, each with its probability — and the function that captures it for discrete variables: the **probability mass function (PMF)**. It is based on the second video of Lesson 2 (Probability Distributions).


## 1. Introduction

Lesson 1 was about calculating *individual* probabilities. Now imagine laying out **all possible scenarios** on a horizontal axis and, for each one, the probability that it happens. That picture is a **probability distribution** — the topic of this whole lesson.

After completing this notebook you will be able to:

- Build the distribution of "number of heads" for 3, 4, and 5 coins by grouping outcomes.
- Define the probability mass function $p(x) = P(X = x)$ and its two requirements.
- Explain *why* middle values are more likely: more outcomes contribute to them.


## 2. Theory

### From outcomes to distribution

For $X$ = number of heads in 3 tosses, group the $2^3 = 8$ equally likely outcomes by their value of $X$:

| $X$ | Outcomes | Count | Probability |
|---|---|---|---|
| 0 | TTT | 1 | 1/8 |
| 1 | HTT, THT, TTH | 3 | 3/8 |
| 2 | HHT, HTH, THH | 3 | 3/8 |
| 3 | HHH | 1 | 1/8 |

One or two heads is much more likely than zero or three — simply because **more outcomes contribute** to those values. The counts 1, 3, 3, 1 divided by 8 are the probabilities, and plotted as bars they form a histogram-like picture: the distribution.

### The probability mass function

For a discrete random variable, the function

$$
p(x) = P(X = x)
$$

is called the **probability mass function (PMF)**, written with a lowercase $p$. It contains *all* the information about how probability distributes over the variable's possible values.

### The two requirements

1. **Non-negativity**: $p(x) \geq 0$ for every $x$ — it is a probability.
2. **Normalization**: $\sum_x p(x) = 1$ — summing over all possible values covers every outcome of the experiment.

Any function satisfying these two conditions is a valid PMF.


## 3. Worked Examples

### Four coins

$2^4 = 16$ outcomes. The counts per value of $X$: **1, 4, 6, 4, 1** — one way for all heads or all tails, four ways for exactly one head (it can be any of the four coins), six ways for two heads.

$$
p(0) = \tfrac{1}{16}, \quad p(1) = \tfrac{4}{16}, \quad p(2) = \tfrac{6}{16}, \quad p(3) = \tfrac{4}{16}, \quad p(4) = \tfrac{1}{16}
$$

### Five coins

$2^5 = 32$ outcomes, with counts **1, 5, 10, 10, 5, 1**:

- all heads: 1 way; exactly one head: 5 ways (the head can be any of the 5 flips);
- exactly two heads: **10 ways** — but how would you *calculate* that 10 without listing? That counting method (combinations) is the subject of the next video.

### The pattern

$X_1$ (3 coins), $X_2$ (4 coins), and $X_3$ (5 coins) are all "number of heads in a fixed number of tosses," and their distributions follow the same shape. A single model covers them all — the **binomial distribution**, coming next.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

rng = np.random.default_rng(seed=42)


### Building PMFs by enumeration

Exactly the video's construction: list every outcome, group by the number of heads, divide by the total.


In [ ]:
def pmf_heads(n_coins):
    outcomes = list(product("HT", repeat=n_coins))
    counts = np.bincount([o.count("H") for o in outcomes], minlength=n_coins + 1)
    return counts, counts / len(outcomes)

for n in [3, 4, 5]:
    counts, pmf = pmf_heads(n)
    print(f"{n} coins: counts {counts.tolist()} of {2**n} -> "
          f"pmf {[f'{p:.4f}' for p in pmf]}")


### Checking the two PMF requirements


In [ ]:
for n in [3, 4, 5]:
    _, pmf = pmf_heads(n)
    print(f"{n} coins: all p(x) >= 0: {bool((pmf >= 0).all())},  "
          f"sum = {pmf.sum():.4f}")


### PMF vs. simulation

The PMF is the exact limit of what a histogram of many experiments estimates.


In [ ]:
n_runs = 100_000
x = (rng.integers(0, 2, size=(n_runs, 5))).sum(axis=1)

_, pmf5 = pmf_heads(5)
print("x   simulated   exact PMF")
for k in range(6):
    print(f"{k}     {(x == k).mean():.4f}     {pmf5[k]:.4f}")


## 5. Visualizations

### The three distributions side by side

The same shape at every size — the pattern the binomial distribution will capture in one formula.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, n in zip(axes, [3, 4, 5]):
    counts, pmf = pmf_heads(n)
    ax.bar(range(n + 1), pmf, color="#4e79a7", edgecolor="white")
    for k, (c, p) in enumerate(zip(counts, pmf)):
        ax.text(k, p + 0.008, f"{c}/{2**n}", ha="center", fontsize=9)
    ax.set_title(f"{n} coins")
    ax.set_xlabel("Number of heads")
    ax.set_xticks(range(n + 1))
    ax.grid(True, alpha=0.3, axis="y")

axes[0].set_ylabel("p(x)")
plt.suptitle("PMFs of the number of heads: counts 1,3,3,1 — 1,4,6,4,1 — 1,5,10,10,5,1")
plt.tight_layout()
plt.show()


### Why the middle wins: stacking the outcomes

Each cell is one of the 8 outcomes of 3 tosses, stacked over its value of $X$ — the video's reordering picture.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

outcomes = ["".join(o) for o in product("HT", repeat=3)]
stacks = {k: [o for o in outcomes if o.count("H") == k] for k in range(4)}

for k, outs in stacks.items():
    for i, o in enumerate(outs):
        ax.add_patch(plt.Rectangle((k - 0.4, i), 0.8, 0.9,
                                   facecolor="#4e79a7", edgecolor="white"))
        ax.text(k, i + 0.45, o, ha="center", va="center",
                color="white", fontsize=12)

ax.set_xlim(-0.7, 3.7)
ax.set_ylim(0, 3.4)
ax.set_xticks(range(4))
ax.set_xlabel("X = number of heads")
ax.set_yticks([])
ax.set_title("8 equally likely outcomes, grouped by X: the middle stacks higher")
plt.show()


## 6. Exercises

### Basic

1. Write the PMF of $X$ = the value of one fair die roll.
2. Is $p(0) = 0.5, p(1) = 0.3, p(2) = 0.3$ a valid PMF? What about $p(0) = 0.9, p(1) = 0.2, p(2) = -0.1$?

### Intermediate

3. For 4 coins, verify from the PMF that $P(X \geq 3) = \frac{5}{16}$.
4. Enumerate the outcomes of rolling **two dice** and build the PMF of $X$ = the sum. Which value has the highest probability, and why?

### Advanced

5. For 6 coins, compute the counts per number of heads by enumeration. Compare the row you get (1, 6, 15, 20, 15, 6, 1) with rows for 3, 4, 5 coins — what famous triangle is emerging?


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  p(x) = 1/6 for x in {1,...,6}, and 0 otherwise.")

# Exercise 2
print("2.  First: sums to 1.1 -> invalid. Second: p(2) < 0 -> invalid.")

# Exercise 3
_, pmf4 = pmf_heads(4)
print(f"3.  P(X>=3) = p(3)+p(4) = {pmf4[3]:.4f}+{pmf4[4]:.4f} = "
      f"{pmf4[3]+pmf4[4]:.4f} = 5/16")

# Exercise 4: sum of two dice
sums = [a + b for a, b in product(range(1, 7), repeat=2)]
counts = np.bincount(sums, minlength=13)[2:]
print("4.  counts for sums 2..12:", counts.tolist())
print("    7 wins: the most pairs (6 of 36) add to 7.")

# Exercise 5: Pascal's triangle
for n in [3, 4, 5, 6]:
    counts, _ = pmf_heads(n)
    print(f"5.  {n} coins: {counts.tolist()}")
print("    These are rows of Pascal's triangle -> binomial coefficients.")


## 8. Conclusion

- A **probability distribution** shows every value a random variable can take together with its probability.
- For discrete variables it is captured by the **PMF** $p(x) = P(X = x)$, which must be non-negative and sum to 1.
- Middle values are likelier because **more outcomes group into them**: 1,3,3,1 (3 coins), 1,4,6,4,1 (4 coins), 1,5,10,10,5,1 (5 coins).
- Counting those ways without listing them — and the single model behind all these coin distributions, the **binomial distribution** — is next.
